# Dunnhumby NGCF seed50/51 — personal Drive resume
Existing completed results and latest checkpoints are preserved. Migration copies missing files only; old packet checkpoints never replace newer durable epochs. Same fixed learning source/run/seed/300epoch/test.


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import json, hashlib, subprocess, sys, importlib.util, urllib.request
import torch
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials,_=google.auth.default()
api=build('drive','v3',credentials=credentials,cache_discovery=False)
"""Bounded Drive retries and account/quota caching, without training RNG use."""
import json
import time


def execute(request, *, upload=False, sleep=time.sleep):
    for attempt in range(6):
        try:
            return request.execute()
        except Exception as exc:
            status = getattr(getattr(exc, 'resp', None), 'status', None)
            try:
                reasons = {r.get('reason') for r in json.loads(exc.content).get('error', {}).get('errors', [])}
            except (ValueError, TypeError, AttributeError):
                reasons = set()
            limited = status == 429 or (status == 403 and bool(reasons & {'rateLimitExceeded', 'userRateLimitExceeded'}))
            # Uploads retry explicit quota rejections only; ambiguous failures stop.
            retry = limited or (not upload and status in (500, 502, 503, 504))
            if not retry or attempt == 5:
                raise
            delay = min(300, 60 * 2 ** attempt)
            print(f'[Drive 대기] HTTP {status}; {delay}초 후 재시도 {attempt+1}/5. 기존 저장본 유지.', flush=True)
            sleep(delay)


class AccountInfo:
    def __init__(self, account, ttl=120, clock=time.monotonic):
        self.account, self.ttl, self.clock = account, ttl, clock
        self.cache = None
        self.api = None

    def get(self, api, force=False):
        if force or self.api is not api or self.cache is None or self.clock()-self.cache[0] >= self.ttl:
            self.cache = None
            info = execute(api.about().get(fields='user(emailAddress),storageQuota'))
            if info['user']['emailAddress'] != self.account:
                raise RuntimeError('개인 Drive 계정 불일치')
            self.cache, self.api = (self.clock(), info), api
        return self.cache[1]

_drive_info = AccountInfo('azsabae@gmail.com')
info=_drive_info.get(api)
assert info['user']['emailAddress']=='azsabae@gmail.com','개인 계정으로 연결하세요.'
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0),'기존과 같은 L4 필요'
SOURCE='63146eff4ed60bfb0cc430b77d58fdc40e4db7c5'
REPO=Path('/content/clv-dh-ngcf-remaining9-'+SOURCE[:12])
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==SOURCE
sys.path.insert(0,str(REPO))
HELPER='46c9c6dbddad7ea384e267f2cbd17708707db994'
for name,sha in {'clv_dh_ngcf_split72':'96c0b3d9a08d2a244c1bc15ea1967fa3414b85f46754cdb69ad9bfeba2b5fb43','clv_ngcf_school_storage':'0d3be22fe7d604adabdf1aaffcfbfe7d1dc748cbba4b1c7073334e0c2bd1bd4c'}.items():
    with urllib.request.urlopen(f'https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/{HELPER}/{name}.py',timeout=60) as response: data=response.read()
    assert hashlib.sha256(data).hexdigest()==sha
    target=Path('/content')/(name+'.py');target.write_bytes(data)
    spec=importlib.util.spec_from_file_location(name,target);module=importlib.util.module_from_spec(spec);sys.modules[name]=module;spec.loader.exec_module(module)
import clv_dh_ngcf_split72 as split
import clv_ngcf_school_storage as storage
storage.ACCOUNT='azsabae@gmail.com'
def personal_space(api,required=0,initial=False):
    info=_drive_info.get(api, force=True)
    if info['user']['emailAddress']!=storage.ACCOUNT: raise RuntimeError('개인 Drive 계정 불일치')
    quota=info['storageQuota']
    if 'limit' not in quota or 'usage' not in quota: raise RuntimeError('개인 Drive 실제 용량 확인 불가')
    available=int(quota['limit'])-int(quota['usage'])
    needed=1_000_000_000+int(required)
    if available<needed: raise storage.InsufficientStorage(available,needed)
    return available
storage.check_space=personal_space
storage.execute = execute
"""Reduce storage-only requests; retain epoch uploads and existing safety checks.

Install only between training calls. No optimizer, RNG or experiment settings
are changed. Space estimates expire after 120s and deduct every attempted
upload without crediting deletions. A low estimate forces a live check.
"""
import time
import hashlib


def install(storage, ttl=120):
    if getattr(storage, '_fast_storage_installed', False):
        return
    original_check = storage.check_space
    cache = {}

    def check(api, required=0, initial=False):
        key = id(api)
        now = time.monotonic()
        entry = cache.get(key)
        needed = max(storage.START_FREE if initial else storage.MIN_FREE,
                     storage.MIN_FREE + int(required))
        if initial or entry is None or now - entry[0] >= ttl or entry[1] < needed:
            cache.pop(key, None)
            available = original_check(api, required=required, initial=initial)
            entry = [time.monotonic(), available]
            cache[key] = entry
        available = entry[1]
        entry[1] -= int(required)  # conservative even when upload fails
        return available

    def publish(api, parent, path, ident, epoch):
        try:
            from googleapiclient.http import MediaFileUpload
            size = path.stat().st_size
            sha, md5 = hashlib.sha256(), hashlib.md5()
            with path.open('rb') as f:
                for block in iter(lambda: f.read(8 * 1024**2), b''):
                    sha.update(block)
                    md5.update(block)
            props = dict(clv_split_run=ident.config_hash, clv_model=ident.model_id,
                         clv_seed=str(ident.seed), clv_epoch=str(epoch), clv_sha256=sha.hexdigest())
            rows = storage.rows_for(api, parent, ident)  # always fresh
            same = [r for r in rows if int(r['appProperties']['clv_epoch']) == epoch]
            if same and any(r['appProperties']['clv_sha256'] != sha.hexdigest() for r in same):
                raise RuntimeError('동일 epoch의 다른 checkpoint: 덮어쓰기 금지')
            if same:
                current = same[0]
            else:
                storage.wait_for_space(api, size)
                # Ambiguous creates are NOT retried; next call lists afresh.
                request = api.files().create(body={
                    'name': f'{ident.model_id}_s{ident.seed}_ep{epoch:03d}.pt',
                    'parents': [parent], 'appProperties': props},
                    media_body=MediaFileUpload(str(path), mimetype='application/octet-stream',
                                               resumable=True), fields=storage.FIELDS)
                current = execute(request, upload=True)
            live = storage.execute(api.files().get(fileId=current['id'], fields=storage.FIELDS))
            for row in (current, live):
                if (row.get('ownedByMe') is not True or row.get('trashed') is not False
                        or row.get('parents') != [parent] or row.get('appProperties') != props
                        or int(row.get('size', -1)) != size or row.get('md5Checksum') != md5.hexdigest()):
                    raise RuntimeError('새 checkpoint 저장 검증 실패: 이전본 보존')
            # Replace redundant second folder listing with the independently
            # verified upload. Unseen concurrent files are never deleted.
            rows = [r for r in rows if r['id'] != live['id']] + [live]
            keep = set(sorted({int(r['appProperties']['clv_epoch']) for r in rows}, reverse=True)[:2])
            for row in rows:
                if int(row['appProperties']['clv_epoch']) in keep:
                    continue
                fresh = storage.execute(api.files().get(fileId=row['id'], fields=storage.FIELDS))
                if any(fresh.get(k) != row.get(k) for k in
                       ('parents', 'appProperties', 'md5Checksum', 'size', 'trashed', 'ownedByMe')):
                    raise RuntimeError('이전 checkpoint 동시변경: 삭제 중단')
                storage.execute(api.files().delete(fileId=row['id']))
            return live
        except BaseException:
            cache.pop(id(api), None)
            raise

    storage.check_space = check
    storage.publish = publish
    storage._fast_storage_installed = True
    print(f'[저장 최적화] 용량 조회 {ttl}초 캐시·업로드 용량 차감. '
          '매epoch 업로드·해시 검증·최신/직전 보존·재시도 유지.', flush=True)

install(storage)
assert storage.START_FREE==storage.MIN_FREE==1_000_000_000
print('개인 계정/L4/고정source 확인. 여유GB:',storage.check_space(api)/1e9)


In [ ]:
from zipfile import ZipFile
import shutil
ARCHIVE_SHA='f94f1af405567582e2f87fd8ab1632e754f3301181cf09fe9e9c8fbb08fc97aa'
archive=Path('/content/drive/MyDrive/dh_ngcf_seed50_51_migrated_20261027/dh_ngcf_school2_migration_20261027.zip')
assert archive.is_file(),'개인 Drive의 이전 폴더에서 이동 ZIP을 확인하세요.'
assert hashlib.sha256(archive.read_bytes()).hexdigest()==ARCHIVE_SHA
ROOT=Path('/content/ngcf_migration_import_20261027')
with ZipFile(archive) as z:
    names=z.namelist()
    assert len(names)==len(set(names)) and all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in names)
    assert z.testzip() is None
    manifest=json.loads(z.read('manifest.json'))
    assert set(names)==set(manifest)|{'manifest.json','source_drive_files.json'}
    for name,sha in manifest.items(): assert hashlib.sha256(z.read(name)).hexdigest()==sha,name
    z.extractall(ROOT)
inputs=Path('/content/ngcf_school2_inputs')
with ZipFile(ROOT/'inputs.zip') as z:
    assert z.testzip() is None
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(inputs)
for name,sha in json.loads((inputs/'manifest.json').read_text()).items():
    assert hashlib.sha256((inputs/name).read_bytes()).hexdigest()==sha
out=Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_ngcf_school2_seed50_51_v1')
run=out/'ae1edba2afd9'
"""Repeatable migration: never replace existing destination files or newer epochs."""
from pathlib import Path
import hashlib
import os
import shutil


def import_missing(source, destination):
    source, destination = Path(source), Path(destination)
    # Check the immutable protocol before copying any missing file.
    protocol = destination / 'protocol.json'
    if protocol.exists() and protocol.read_bytes() != (source/'protocol.json').read_bytes():
        raise RuntimeError('Existing run protocol differs; nothing imported')
    copied, preserved = [], []
    for f in sorted(source.rglob('*')):
        if not f.is_file():
            continue
        rel = f.relative_to(source)
        dest = destination / rel
        if dest.exists():
            if not dest.is_file():
                raise RuntimeError(f'Expected file: {dest}')
            preserved.append(str(rel))
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        # Exclusive creation never overwrites a file created concurrently.
        try:
            with dest.open('xb') as out, f.open('rb') as inp:
                shutil.copyfileobj(inp, out)
                out.flush()
                os.fsync(out.fileno())
        except FileExistsError:
            preserved.append(str(rel))
            continue
        if hashlib.sha256(dest.read_bytes()).digest() != hashlib.sha256(f.read_bytes()).digest():
            raise RuntimeError(f'Imported file hash mismatch: {dest}')
        copied.append(str(rel))
    return {'copied': copied, 'preserved': preserved}


def pending_epochs(remote_rows, archive_rows, *, completed):
    """Never republish an old packet over a resumed run; retain remote top two."""
    if completed:
        return []
    current = max((int(r['appProperties']['clv_epoch']) for r in remote_rows), default=0)
    for row in archive_rows:
        p = row['appProperties']
        for live in remote_rows:
            q = live['appProperties']
            if int(p['clv_epoch']) == int(q['clv_epoch']) and p['clv_sha256'] != q['clv_sha256']:
                raise RuntimeError('Same epoch has different checkpoint bytes; originals preserved')
    return sorted([r for r in archive_rows if int(r['appProperties']['clv_epoch']) > current],
                  key=lambda r: int(r['appProperties']['clv_epoch']))

import_report=import_missing(ROOT/'run',run)
print('Missing files imported:',len(import_report['copied']),'Existing files preserved:',len(import_report['preserved']))


In [ ]:
import clv_dh_ngcf_remaining9 as screen
from dataclasses import asdict
assert Path(screen.__file__).resolve().parent==REPO.resolve()
assert Path(screen.original.__file__).resolve().parent==REPO.resolve()
screen.base.v3.SCHEMA['dunnhumby']['tx_path']=str(inputs/'raw/dunnhumby/transaction_data.csv')
screen.base.v3.SCHEMA['dunnhumby']['item_meta_path']=str(inputs/'raw/dunnhumby/product.csv')
cfg=screen.configure(out_dir=str(out),m2_reference=str(inputs/'references/clv_m2_adaptive_dunnhumby_final10_results.zip'),m4_reference=str(inputs/'references/clv_m4_user_centered_final10_results.zip'),seed49_reference=str(inputs/'references/dh_ngcf_final8_seed49_results.zip'))
prepared=screen.prepare(cfg)
assert prepared['config_hash']=='ae1edba2afd9'
assert prepared['protocol']==json.loads((ROOT/'run/protocol.json').read_text())
split.check_personal_receipt(prepared,ROOT/'personal7_assignment.json')
split.install(prepared,'school')
for seed in (50,51):
    for label in screen.ORDER:
        screen.completed(prepared,label,seed)  # validates existing completed identities/results
print('Current completion:',split.status(prepared,'school'))
checkpoint_folder=storage.install(prepared,api)
source_rows=json.loads((ROOT/'source_drive_files.json').read_text())
assert len(source_rows)==6
from collections import defaultdict
groups=defaultdict(list)
for row in source_rows:
    props=row['appProperties'];groups[(props['clv_model'],int(props['clv_seed']))].append(row)
for (model_id,seed), packet_rows in groups.items():
    label=model_id.removeprefix('ngcf_').removesuffix('_lastweek').upper()
    ident=screen.identity(prepared,label,seed)
    live=storage.rows_for(api,checkpoint_folder,ident)
    done=screen.completed(prepared,label,seed) is not None
    if not live and not done:
        for stage in (run/'progress'/'stages').glob('*.json'):
            record=json.loads(stage.read_text())
            if record.get('model_id')==model_id and record.get('seed')==seed:
                assert int(record.get('epoch',0))<=max(int(r['appProperties']['clv_epoch']) for r in packet_rows),'Newer progress without durable checkpoint; do not import stale packet'
    for row in pending_epochs(live,packet_rows,completed=done):
        props=row['appProperties'];ep=int(props['clv_epoch']);f=ROOT/'checkpoints'/row['name']
        assert hashlib.sha256(f.read_bytes()).hexdigest()==props['clv_sha256']
        ck=torch.load(f,map_location='cpu',weights_only=False)
        assert ck['identity']==asdict(ident) and ck['epoch']==ep
        assert [h['epoch'] for h in ck['history']]==list(range(1,ep+1))
        assert all(k in ck for k in ['model_state','optimizer_state','numpy_rng_state','torch_rng_state','cuda_rng_state'])
        del ck
        storage.publish(api,checkpoint_folder,f,ident,ep)
    print(model_id,seed,'completed preserved' if done else 'latest durable checkpoints preserved', [int(r['appProperties']['clv_epoch']) for r in storage.rows_for(api,checkpoint_folder,ident)])
print('Migration verified. Completed models reused; unfinished models resume from latest verified Drive epoch.')


In [ ]:
# seed50/51 only; completed models are reused and latest durable state restored.
assert _drive_info.get(api)['user']['emailAddress']=='azsabae@gmail.com'
try:
    result=split.run(prepared,'school')
except Exception:
    import traceback
    traceback.print_exc()
    print('저장 확인 실패로 중단. 기존 결과와 최신/직전 checkpoint 보존.', flush=True)
else:
    packet=split.export_partition(prepared,'school','/content/drive/MyDrive/dh_ngcf_school2_results.zip')
    print('50/51 완료 결과 패킷:',packet)
    print(json.dumps(result,ensure_ascii=False,indent=2))
